# Article-style BiLSTM Recreation

This notebook starts from `data_clean/all_fights.csv` and recreates the main steps described in the article as closely as possible, skipping weight because it is not in the current raw dataset.

It tests two versions:

1. **Article-style same-fight features**: uses the fight's recorded stats from `all_fights.csv`. This can produce inflated accuracy because these stats are known only after the fight.
2. **Historical pre-fight features**: reconstructs fighter history from earlier fights only. This is the realistic prediction setup.

Validation uses your preferred method: **5 random 90/10 stratified splits**.

In [4]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False

try:
    import tensorflow as tf
    from tensorflow.keras import layers, models, callbacks
    HAS_TENSORFLOW = True
except Exception:
    HAS_TENSORFLOW = False

RANDOM_STATE = 42
SPLIT_SEEDS = [11, 22, 33, 44, 55]
TEST_SIZE = 0.10

ARTICLE_START = "2012-12-01"
ARTICLE_END = "2022-12-31"
TOP_N_FEATURES = 10

warnings.filterwarnings("ignore", category=FutureWarning)

print("xgboost available:", HAS_XGBOOST)
print("tensorflow available:", HAS_TENSORFLOW)

xgboost available: True
tensorflow available: False


## Load And Clean Data

The article uses UFCStats data from roughly 2012-2022, fills missing values with 0, removes outliers, builds fighter portrait variables, and then performs feature selection.

In [5]:
DATA_PATH = Path("..") / "data_clean" / "all_fights.csv"
raw = pd.read_csv(DATA_PATH)

df = raw.copy()
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["winner_side"] = np.select(
    [df["winner"].eq(df["R_fighter"]), df["winner"].eq(df["B_fighter"])],
    ["R", "B"],
    default="",
)
df["winner_side"] = df["winner_side"].replace("", np.nan)
df = df.dropna(subset=["date", "winner_side"]).copy()
df = df[(df["date"] >= ARTICLE_START) & (df["date"] <= ARTICLE_END)].copy()
df = df.sort_values(["date", "R_fighter", "B_fighter"]).reset_index(drop=True)
df["fight_id"] = np.arange(len(df))

def parse_height_inches(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    match = re.search(r"(\d+)\s*'\s*(\d+)", text)
    if match:
        return int(match.group(1)) * 12 + int(match.group(2))
    return pd.to_numeric(text, errors="coerce")

def parse_reach_inches(value):
    if pd.isna(value):
        return np.nan
    return pd.to_numeric(str(value).replace('"', '').strip(), errors="coerce")

def parse_time_seconds(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    if not text or text == "--":
        return np.nan
    parts = text.split(":")
    if len(parts) != 2:
        return np.nan
    return int(parts[0]) * 60 + int(parts[1])

for side in ["R", "B"]:
    df[f"{side}_DOB_dt"] = pd.to_datetime(df[f"{side}_DOB"], errors="coerce")
    df[f"{side}_age"] = (df["date"] - df[f"{side}_DOB_dt"]).dt.days / 365.25
    df[f"{side}_Height_inches"] = df[f"{side}_Height"].map(parse_height_inches)
    df[f"{side}_Reach_inches"] = df[f"{side}_Reach"].map(parse_reach_inches)
    df[f"{side}_CTRL_seconds"] = df[f"{side}_CTRL"].map(parse_time_seconds)
    df[f"{side}_Stance"] = df[f"{side}_Stance"].fillna("Unknown")

numeric_suffixes = [
    "KD", "SIG_STR_landed", "SIG_STR_att", "SIG_STR_pct", "TOTAL_STR_landed", "TOTAL_STR_att",
    "TD_landed", "TD_att", "TD_pct", "SUB_ATT", "REV", "HEAD_landed", "HEAD_att",
    "BODY_landed", "BODY_att", "LEG_landed", "LEG_att", "DISTANCE_landed", "DISTANCE_att",
    "CLINCH_landed", "CLINCH_att", "GROUND_landed", "GROUND_att", "CTRL_seconds",
]

for side in ["R", "B"]:
    for suffix in numeric_suffixes:
        col = f"{side}_{suffix}"
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

df["DIFF_age"] = df["R_age"] - df["B_age"]
df["DIFF_height"] = df["R_Height_inches"] - df["B_Height_inches"]
df["DIFF_reach"] = df["R_Reach_inches"] - df["B_Reach_inches"]
df["stance_matchup"] = df["R_Stance"].astype(str) + "_vs_" + df["B_Stance"].astype(str)

def drop_iqr_outliers(data, cols, multiplier=1.5):
    keep = pd.Series(True, index=data.index)
    for col in cols:
        series = pd.to_numeric(data[col], errors="coerce")
        q1, q3 = series.quantile([0.25, 0.75])
        iqr = q3 - q1
        if pd.isna(iqr) or iqr == 0:
            continue
        keep &= series.between(q1 - multiplier * iqr, q3 + multiplier * iqr) | series.isna()
    return data.loc[keep].copy()

physical_cols = ["R_age", "B_age", "R_Height_inches", "B_Height_inches", "R_Reach_inches", "B_Reach_inches"]
df_clean = drop_iqr_outliers(df, physical_cols).reset_index(drop=True)

print("Loaded:", DATA_PATH)
print("Article-period rows before outlier removal:", len(df))
print("Rows after physical outlier removal:", len(df_clean))
print(df_clean["winner_side"].value_counts())
print("Majority baseline:", df_clean["winner_side"].value_counts(normalize=True).max().round(4))

Loaded: ..\data_clean\all_fights.csv
Article-period rows before outlier removal: 4728
Rows after physical outlier removal: 4631
winner_side
R    2699
B    1932
Name: count, dtype: int64
Majority baseline: 0.5828


## Shared Modeling Helpers

The article uses XGBoost feature importance and a small BiLSTM. If `xgboost` or `tensorflow` is not installed, this notebook falls back to a Random Forest feature selector and an MLP classifier so the experiment still runs.

In [6]:
def make_xy(feature_frame, target):
    X = feature_frame.copy()
    X = pd.get_dummies(X, dummy_na=True)
    X = X.replace([np.inf, -np.inf], np.nan)
    y = target.map({"B": 0, "R": 1}).astype(int)
    return X, y

def select_top_features(X, y, top_n=10, random_state=42):
    X_model = X.copy()
    X_model = pd.DataFrame(
        SimpleImputer(strategy="constant", fill_value=0).fit_transform(X_model),
        columns=X_model.columns,
        index=X_model.index,
    )
    if HAS_XGBOOST:
        selector = XGBClassifier(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric="logloss",
            random_state=random_state,
            n_jobs=-1,
        )
    else:
        selector = RandomForestClassifier(
            n_estimators=500,
            min_samples_leaf=5,
            max_features="sqrt",
            random_state=random_state,
            n_jobs=-1,
        )
    selector.fit(X_model, y)
    importances = pd.Series(selector.feature_importances_, index=X_model.columns).sort_values(ascending=False)
    selected = importances.head(top_n)
    return selected

def apply_article_feature_weights(X, selected_importances):
    selected_cols = selected_importances.index.tolist()
    X_selected = X[selected_cols].copy()
    weights = selected_importances / selected_importances.max()
    return X_selected.mul(weights, axis=1)

def build_bilstm(n_features, seed):
    tf.keras.utils.set_random_seed(seed)
    model = models.Sequential([
        layers.Input(shape=(n_features, 1)),
        layers.Bidirectional(layers.LSTM(16)),
        layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer="RMSprop", loss="binary_crossentropy", metrics=["accuracy"])
    return model

def evaluate_five_splits(X, y, label, use_bilstm=True):
    rows = []
    last_confusion = None
    for i, seed in enumerate(SPLIT_SEEDS, start=1):
        splitter = StratifiedShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=seed)
        train_idx, test_idx = next(splitter.split(X, y))
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        imputer = SimpleImputer(strategy="constant", fill_value=0)
        scaler = StandardScaler()
        X_train_arr = scaler.fit_transform(imputer.fit_transform(X_train))
        X_test_arr = scaler.transform(imputer.transform(X_test))

        if use_bilstm and HAS_TENSORFLOW:
            model = build_bilstm(X_train_arr.shape[1], seed)
            early_stop = callbacks.EarlyStopping(
                monitor="val_loss",
                patience=10,
                restore_best_weights=True,
            )
            model.fit(
                X_train_arr.reshape(-1, X_train_arr.shape[1], 1),
                y_train,
                validation_split=0.1,
                epochs=100,
                batch_size=32,
                callbacks=[early_stop],
                verbose=0,
            )
            proba = model.predict(X_test_arr.reshape(-1, X_test_arr.shape[1], 1), verbose=0).ravel()
        else:
            model = MLPClassifier(
                hidden_layer_sizes=(32, 16),
                activation="relu",
                solver="adam",
                max_iter=500,
                early_stopping=True,
                random_state=seed,
            )
            model.fit(X_train_arr, y_train)
            proba = model.predict_proba(X_test_arr)[:, 1]

        pred = (proba >= 0.5).astype(int)
        acc = accuracy_score(y_test, pred)
        auc = roc_auc_score(y_test, proba)
        last_confusion = confusion_matrix(y_test, pred)
        rows.append({"feature_set": label, "split": i, "seed": seed, "accuracy": acc, "roc_auc": auc})

    split_results = pd.DataFrame(rows)
    summary = pd.DataFrame([{
        "feature_set": label,
        "model": "BiLSTM" if use_bilstm and HAS_TENSORFLOW else "MLP fallback",
        "n_features": X.shape[1],
        "mean_accuracy": split_results["accuracy"].mean(),
        "std_accuracy": split_results["accuracy"].std(),
        "mean_auc": split_results["roc_auc"].mean(),
        "std_auc": split_results["roc_auc"].std(),
    }])
    return summary, split_results, last_confusion

## Experiment 1: Article-style Same-fight Features

This uses the same row's strike, takedown, control, and body/leg/head/distance stats. This is likely closest to the paper's vague feature table, but it is not a valid pre-fight predictor.

In [7]:
same_fight_cols = []
for side in ["R", "B"]:
    same_fight_cols.extend([
        f"{side}_age", f"{side}_Height_inches", f"{side}_Reach_inches", f"{side}_Stance",
    ])
    same_fight_cols.extend([f"{side}_{suffix}" for suffix in numeric_suffixes if f"{side}_{suffix}" in df_clean.columns])

same_fight_cols.extend(["DIFF_age", "DIFF_height", "DIFF_reach", "stance_matchup"])
same_fight_features = df_clean[same_fight_cols].copy()
X_same, y_same = make_xy(same_fight_features, df_clean["winner_side"])

same_importances = select_top_features(X_same, y_same, top_n=TOP_N_FEATURES)
X_same_top = apply_article_feature_weights(X_same, same_importances)

print("Same-fight candidate features:", X_same.shape[1])
print("Selected article-style top features:")
display(same_importances.to_frame("importance"))

Same-fight candidate features: 82
Selected article-style top features:


,importance
B_GROUND_landed,0.122194
B_KD,0.083302
R_GROUND_landed,0.082421
R_KD,0.051714
B_TOTAL_STR_landed,0.043445
B_SUB_ATT,0.035182
R_GROUND_att,0.033833
R_TOTAL_STR_landed,0.031947
R_SUB_ATT,0.031889
R_SIG_STR_pct,0.029144


In [8]:
same_summary, same_splits, same_confusion = evaluate_five_splits(
    X_same_top,
    y_same,
    "article-style same-fight top 10",
    use_bilstm=True,
)

display(same_summary)
display(same_splits)
print("Last split confusion matrix:")
print(same_confusion)

,feature_set,model,n_features,mean_accuracy,std_accuracy,mean_auc,std_auc
0,article-style same-fight top 10,MLP fallback,10,0.844397,0.013233,0.92824,0.004694


,feature_set,split,seed,accuracy,roc_auc
0,article-style same-fight top 10,1,11,0.855603,0.935605
1,article-style same-fight top 10,2,22,0.855603,0.928656
2,article-style same-fight top 10,3,33,0.836207,0.926384
3,article-style same-fight top 10,4,44,0.825431,0.922757
4,article-style same-fight top 10,5,55,0.849138,0.927797


Last split confusion matrix:
[[158  36]
 [ 34 236]]


## Experiment 2: Historical Pre-fight Features

This rebuilds article-like historical features from `all_fights.csv`, but only using fights before the current fight. This avoids same-fight leakage.

In [9]:
def build_long_fighter_rows(data):
    frames = []
    for side, opp in [("R", "B"), ("B", "R")]:
        cols = {
            "fight_id": data["fight_id"],
            "date": data["date"],
            "fighter": data[f"{side}_fighter"],
            "side": side,
            "won": (data["winner_side"] == side).astype(int),
            "age": data[f"{side}_age"],
            "height": data[f"{side}_Height_inches"],
            "reach": data[f"{side}_Reach_inches"],
        }
        for suffix in numeric_suffixes:
            col = f"{side}_{suffix}"
            if col in data.columns:
                cols[suffix] = data[col]
        frames.append(pd.DataFrame(cols))
    long = pd.concat(frames, ignore_index=True)
    return long.sort_values(["date", "fight_id", "side"]).reset_index(drop=True)

long = build_long_fighter_rows(df_clean)
grouped = long.groupby("fighter", sort=False)
long["prior_fights"] = grouped.cumcount()
long["prior_wins"] = grouped["won"].cumsum() - long["won"]
long["prior_win_pct"] = long["prior_wins"] / long["prior_fights"].replace(0, np.nan)

history_stat_cols = [c for c in numeric_suffixes if c in long.columns]
for col in history_stat_cols:
    prior_sum = grouped[col].cumsum() - long[col].fillna(0)
    long[f"hst_{col}_avg"] = prior_sum / long["prior_fights"].replace(0, np.nan)

long["hst_SIG_STR_pct"] = long["hst_SIG_STR_landed_avg"] / long["hst_SIG_STR_att_avg"].replace(0, np.nan)
long["hst_TD_pct"] = long["hst_TD_landed_avg"] / long["hst_TD_att_avg"].replace(0, np.nan)

hist_feature_cols = [
    "fight_id", "side", "prior_fights", "prior_win_pct", "hst_SIG_STR_pct", "hst_TD_pct",
] + [f"hst_{c}_avg" for c in history_stat_cols]

hist = long[hist_feature_cols].copy()
hist_R = hist[hist["side"] == "R"].drop(columns="side").add_prefix("R_").rename(columns={"R_fight_id": "fight_id"})
hist_B = hist[hist["side"] == "B"].drop(columns="side").add_prefix("B_").rename(columns={"B_fight_id": "fight_id"})

pre = df_clean[[
    "fight_id", "winner_side", "R_age", "B_age", "R_Height_inches", "B_Height_inches",
    "R_Reach_inches", "B_Reach_inches", "R_Stance", "B_Stance", "stance_matchup",
    "DIFF_age", "DIFF_height", "DIFF_reach",
]].merge(hist_R, on="fight_id", how="left").merge(hist_B, on="fight_id", how="left")

pre["DIFF_prior_fights"] = pre["R_prior_fights"] - pre["B_prior_fights"]
pre["DIFF_prior_win_pct"] = pre["R_prior_win_pct"] - pre["B_prior_win_pct"]
pre["DIFF_hst_SIG_STR_pct"] = pre["R_hst_SIG_STR_pct"] - pre["B_hst_SIG_STR_pct"]
pre["DIFF_hst_TD_pct"] = pre["R_hst_TD_pct"] - pre["B_hst_TD_pct"]

for c in history_stat_cols:
    pre[f"DIFF_hst_{c}_avg"] = pre[f"R_hst_{c}_avg"] - pre[f"B_hst_{c}_avg"]

historical_feature_cols = [c for c in pre.columns if c not in ["fight_id", "winner_side"]]
historical_features = pre[historical_feature_cols].copy()
X_hist, y_hist = make_xy(historical_features, pre["winner_side"])

hist_importances = select_top_features(X_hist, y_hist, top_n=TOP_N_FEATURES)
X_hist_top = apply_article_feature_weights(X_hist, hist_importances)

print("Historical candidate features:", X_hist.shape[1])
print("Selected historical top features:")
display(hist_importances.to_frame("importance"))

Historical candidate features: 118
Selected historical top features:


,importance
DIFF_age,0.030820
B_Reach_inches,0.024768
R_prior_win_pct,0.022222
R_Reach_inches,0.020341
B_hst_SIG_STR_landed_avg,0.018424
B_hst_LEG_landed_avg,0.014350
B_hst_TOTAL_STR_att_avg,0.013679
DIFF_hst_LEG_att_avg,0.012948
DIFF_hst_GROUND_att_avg,0.012693
B_hst_SIG_STR_pct_avg,0.012675


In [10]:
hist_summary, hist_splits, hist_confusion = evaluate_five_splits(
    X_hist_top,
    y_hist,
    "historical pre-fight top 10",
    use_bilstm=True,
)

display(hist_summary)
display(hist_splits)
print("Last split confusion matrix:")
print(hist_confusion)

,feature_set,model,n_features,mean_accuracy,std_accuracy,mean_auc,std_auc
0,historical pre-fight top 10,MLP fallback,10,0.621983,0.015881,0.649309,0.046014


,feature_set,split,seed,accuracy,roc_auc
0,historical pre-fight top 10,1,11,0.627155,0.702425
1,historical pre-fight top 10,2,22,0.622845,0.640473
2,historical pre-fight top 10,3,33,0.635776,0.683601
3,historical pre-fight top 10,4,44,0.629310,0.635586
4,historical pre-fight top 10,5,55,0.594828,0.584460


Last split confusion matrix:
[[ 41 153]
 [ 35 235]]


## Combined Result

If the same-fight result is much higher than the historical pre-fight result, that is evidence that the article's high accuracy likely came from post-fight/stat leakage or unclear validation.

In [11]:
combined_summary = pd.concat([same_summary, hist_summary], ignore_index=True).sort_values("mean_accuracy", ascending=False)
display(combined_summary)

combined_splits = pd.concat([same_splits, hist_splits], ignore_index=True)
display(combined_splits.pivot(index="split", columns="feature_set", values="accuracy"))

,feature_set,model,n_features,mean_accuracy,std_accuracy,mean_auc,std_auc
0,article-style same-fight top 10,MLP fallback,10,0.844397,0.013233,0.928240,0.004694
1,historical pre-fight top 10,MLP fallback,10,0.621983,0.015881,0.649309,0.046014


feature_set,article-style same-fight top 10,historical pre-fight top 10
split,,
1,0.855603,0.627155
2,0.855603,0.622845
3,0.836207,0.635776
4,0.825431,0.629310
5,0.849138,0.594828
